# Лабораторная работа 5, Самсонов Савелий Артёмович М8О-406Б-21

### Выбор задач и датасетов

Киноиндустрия сталкивается с серьезной проблемой при прогнозировании успеха фильмов на конкурентном рынке. Понимание ключевых факторов, влияющих на это и на получение дохода от фильма, имеет решающее значение для продюсеров, студий и инвесторов при принятии стратегических решений. Поэтому для определения переменных, влияющих на успех фильма, таких как бюджет, маркетинговые расходы, продолжительность фильма и рейтинги главных актеров, режиссеров и критиков, необходимо прогностическое моделирование. Для обучения моделей, которые могут предсказать успешность фильмов в прокате, взяты датасеты Movie_classification.csv и Movie_regression.xls, опубликованные на kaggle.

### Выбор метрик для классификации

1. Accuracy, измеряет долю правильно классифицированных экземпляров от общего числа примеров. Простая и понятная метрика, но может давать ошибочное представление для несбалансированных классов (когда классы имеют существенно отличающееся количество экземпляров).
2. Precision, измеряет долю правильных положительных предсказаний среди всех предсказанных положительных примеров.
3. Recall, измеряет долю правильно предсказанных положительных примеров среди всех реальных положительных примеров.
Precision, Recall важны в случае несбалансированных классов и при необходимости минимизировать ложные срабатывания.
4. F1-мера, является гармоническим средним между точностью и полнотой и используется для сбалансирования этих двух метрик. Комбинирует точность и полноту, идеально подходит для несбалансированных задач.

### Выбор метрик для регрессии

1. R², отношение между суммой квадратов отклонений предсказанных значений от среднего значения и суммой квадратов отклонений истинных значений от среднего. Показывает, какая доля вариации в целевой переменной объясняется моделью. Хороший показатель R² близкий к 1 означает, что модель хорошо объясняет данные, однако для некоторых типов задач (например, с незначительными отклонениями) значение R² может быть не таким информативным.
2. MAE, измеряет среднее абсолютное отклонение между предсказанными и истинными значениями. Полезна, когда важно понять, насколько в среднем модель ошибается по величине предсказанных значений. MAE не так чувствительна к выбросам, как другие метрики.
3. MSE, измеряет средний квадрат разницы между предсказанными и истинными значениями. MSE часто используется, когда важно акцентировать внимание на больших ошибках. Более чувствительна к выбросам, чем MAE, и может быть полезна, когда крупные ошибки особенно нежелательны.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
import os
dataset_path = '.\\input'

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

.\input\Movie_classification.csv
.\input\Movie_regression.xls


## 2. Создание бейзлайна и оценка качества

### Обучение модели из sklearn (для классификации) и оценка качества по выбранным метрикам

Загрузим датасет

In [44]:
df = pd.read_csv(dataset_path + "\\Movie_classification.csv")
df

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection,Start_Tech_Oscar
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000,1
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200,0
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400,1
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800,1
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800,0
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200,0
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800,0
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000,0


Удалим некоторые параметры

In [45]:
if "Genre" in df:
  del df["Genre"]
if "3D_available" in df:
  del df["3D_available"]
if "Time_taken" in df:
  del df["Time_taken"]

Просмотрим информацию о значениях полей и убедимся, что все они допустимы

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  Twitter_hastags      506 non-null    float64
 12  Avg_age_actors       506 non-null    int64  
 13  Num_multiplex        506 non-null    int64  
 14  Collection           506 non-null    int64  
 15  Start_Tech_Oscar     506 non-null    int

Создадим выборки для обучения и тестирования

In [48]:
X1 = df.drop('Start_Tech_Oscar', axis=1)
y1 = df['Start_Tech_Oscar']

X1_train,X1_test,y1_train,y1_test = train_test_split(X1.values, y1.values, random_state = 0)

Обучение модели для классификации

In [49]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

sk_gradboost_clf = GradientBoostingClassifier()
sk_gradboost_clf.fit(X1_train, y1_train)
sk_gradboost_clf_pred_res = sk_gradboost_clf.predict(X1_test)
sk_gradboost_clf_accuracy = accuracy_score(y1_test, sk_gradboost_clf_pred_res)
sk_gradboost_clf_precision = precision_score(y1_test, sk_gradboost_clf_pred_res)
sk_gradboost_clf_recall = recall_score(y1_test, sk_gradboost_clf_pred_res)
sk_gradboost_clf_f1 = f1_score(y1_test, sk_gradboost_clf_pred_res)

print(f'sk gradboost classifier accuracy: {sk_gradboost_clf_accuracy:}')
print(f'sk gradboost classifier precision: {sk_gradboost_clf_precision:}')
print(f'sk gradboost classifier recall: {sk_gradboost_clf_recall:}')
print(f'sk gradboost classifier f1: {sk_gradboost_clf_f1:}')

sk gradboost classifier accuracy: 0.5511811023622047
sk gradboost classifier precision: 0.6376811594202898
sk gradboost classifier recall: 0.5789473684210527
sk gradboost classifier f1: 0.6068965517241379


### Обучение модели из sklearn (для регрессии) и оценка качества по выбранным метрикам

Загрузим датасет

In [50]:
df2 = pd.read_csv(dataset_path + "\\Movie_regression.xls")
df2

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000


Просмотрим информацию о значениях полей для проверки, что все они допустимы

In [51]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  3D_available         506 non-null    object 
 12  Time_taken           494 non-null    float64
 13  Twitter_hastags      506 non-null    float64
 14  Genre                506 non-null    object 
 15  Avg_age_actors       506 non-null    int

Удалим некоторые параметры

In [52]:
del df2['3D_available']
del df2['Genre']
del df2['Time_taken']
df2.head()

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,Twitter_hastags,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,223.840,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,243.456,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,2022.400,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,225.344,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,225.792,55,395,72400


Создадим выборки для обучения и тестирования

In [53]:
X2 = df2.drop('Collection', axis=1)
y2 = df2['Collection']

X2_train,X2_test,y2_train,y2_test = train_test_split(X2.values, y2.values, random_state = 42)

Обучение модели для регрессии

In [54]:
sk_gradboost_reg = GradientBoostingRegressor()
sk_gradboost_reg.fit(X2_train, y2_train)
sk_gradboost_reg_pred_res = sk_gradboost_reg.predict(X2_test)
sk_gradboost_reg_r2 = r2_score(y2_test, sk_gradboost_reg_pred_res)
sk_gradboost_reg_mae = mean_absolute_error(y2_test, sk_gradboost_reg_pred_res)
sk_gradboost_reg_mse = mean_squared_error(y2_test, sk_gradboost_reg_pred_res)

print(f'sk gradboost regressor r2: {sk_gradboost_reg_r2}')
print(f'sk gradboost regressor mae: {sk_gradboost_reg_mae}')
print(f'sk gradboost regressor mse: {sk_gradboost_reg_mse}')

sk gradboost regressor r2: 0.8758104195019669
sk gradboost regressor mae: 4123.960914906194
sk gradboost regressor mse: 34998633.4241566


## 3. Улучшение бейзлайна

### Сформулировать гипотезы (препроцессинг данных, визуализация данных, формирование новых признаков, подбор гиперпараметров на кросс-валидации и т.д.)

1. Формирование новых признаков: для задачи классификации можно создать новые признаки, комбинирующие в себе расходы и рейтинг участвующих в создании фильма людей соответственно.
2. Масштабирование данных во время предобработки
3. Подбор гиперпараметров
    - Количество деревьев n_estimators
    - Максимальная глубина деревьев max_depth
    - Скорость обучения learning_rate

#### Задача классификации

1. Формирование новых признаков

In [55]:
data3 = df.copy()
data3["Expense"] = data3["Marketing expense"] + data3["Production expense"]
data3["Rating"] = data3["Lead_ Actor_Rating"] + data3["Lead_Actress_rating"] + data3["Director_rating"] + data3["Producer_rating"]

In [56]:
data4 = data3.copy()
data4 = data4.drop(["Marketing expense","Production expense","Lead_ Actor_Rating","Lead_Actress_rating","Director_rating","Producer_rating"],axis = 1)

In [57]:
X1_new = data4.drop('Start_Tech_Oscar', axis=1)
y1_new = data4['Start_Tech_Oscar']

X1_train_new,X1_test_new,y1_train_new,y1_test_new = train_test_split(X1_new.values, y1_new.values, random_state = 0)

Проверим, как повлияло введение новых признаков

In [58]:
sk_gradboost_clf = GradientBoostingClassifier()
sk_gradboost_clf.fit(X1_train_new, y1_train_new)
sk_gradboost_clf_pred_res = sk_gradboost_clf.predict(X1_test_new)
sk_gradboost_clf_accuracy = accuracy_score(y1_test_new, sk_gradboost_clf_pred_res)
sk_gradboost_clf_precision = precision_score(y1_test_new, sk_gradboost_clf_pred_res)
sk_gradboost_clf_recall = recall_score(y1_test_new, sk_gradboost_clf_pred_res)
sk_gradboost_clf_f1 = f1_score(y1_test_new, sk_gradboost_clf_pred_res)

print(f'sk gradboost classifier accuracy: {sk_gradboost_clf_accuracy:}')
print(f'sk gradboost classifier precision: {sk_gradboost_clf_precision:}')
print(f'sk gradboost classifier recall: {sk_gradboost_clf_recall:}')
print(f'sk gradboost classifier f1: {sk_gradboost_clf_f1:}')

sk gradboost classifier accuracy: 0.5984251968503937
sk gradboost classifier precision: 0.6923076923076923
sk gradboost classifier recall: 0.5921052631578947
sk gradboost classifier f1: 0.6382978723404255


2. Попробуем добавить ко введению новых признаков Масштабирование данных

In [60]:
# scaler = StandardScaler()
# X1_train_scaled = scaler.fit_transform(X1_train)
# X1_test_scaled = scaler.transform(X1_test)

scaler = StandardScaler()
X1_train_new_scaled = scaler.fit_transform(X1_train_new)
X1_test_new_scaled = scaler.transform(X1_test_new)

In [61]:
sk_gradboost_clf = GradientBoostingClassifier()
sk_gradboost_clf.fit(X1_train_new_scaled, y1_train)
sk_gradboost_clf_pred_res = sk_gradboost_clf.predict(X1_test_new_scaled)
sk_gradboost_clf_accuracy = accuracy_score(y1_test_new, sk_gradboost_clf_pred_res)
sk_gradboost_clf_precision = precision_score(y1_test_new, sk_gradboost_clf_pred_res)
sk_gradboost_clf_recall = recall_score(y1_test_new, sk_gradboost_clf_pred_res)
sk_gradboost_clf_f1 = f1_score(y1_test_new, sk_gradboost_clf_pred_res)

print(f'sk gradboost classifier accuracy: {sk_gradboost_clf_accuracy:}')
print(f'sk gradboost classifier precision: {sk_gradboost_clf_precision:}')
print(f'sk gradboost classifier recall: {sk_gradboost_clf_recall:}')
print(f'sk gradboost classifier f1: {sk_gradboost_clf_f1:}')

sk gradboost classifier accuracy: 0.5984251968503937
sk gradboost classifier precision: 0.6923076923076923
sk gradboost classifier recall: 0.5921052631578947
sk gradboost classifier f1: 0.6382978723404255


3. Попробуем добавить ко введению новых признаков Подбор гиперпараметров

In [68]:
from sklearn.model_selection import GridSearchCV

param_grid_class = {
    'n_estimators': [10, 50, 100, 150],
    'max_depth': [None, 4, 10, 20, 30],
    'learning_rate': [0.1, 0.25, 0.5, 0.75]
}

gradboost_class = GradientBoostingClassifier()
grid_search_class = GridSearchCV(gradboost_class, param_grid_class, cv=5, scoring='accuracy')
grid_search_class.fit(X1_train_new, y1_train_new)
best_gradboost_class = grid_search_class.best_estimator_

print("Лучшие параметры для классификации:", grid_search_class.best_params_)

y_pred_class = best_gradboost_class.predict(X1_test_new)
accuracy = accuracy_score(y1_test_new, y_pred_class)
precision = precision_score(y1_test_new, y_pred_class)
recall = recall_score(y1_test_new, y_pred_class)
f1 = f1_score(y1_test_new, y_pred_class)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Лучшие параметры для классификации (HAR): {'learning_rate': 0.75, 'max_depth': 4, 'n_estimators': 150}
Accuracy: 0.6141732283464567
Precision: 0.6901408450704225
Recall: 0.6447368421052632
F1 Score: 0.6666666666666666


#### Задача регрессии

2. Масштабирование данных

In [21]:
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

In [63]:
sk_gradboost_reg = GradientBoostingRegressor()
sk_gradboost_reg.fit(X2_train_scaled, y2_train)
sk_gradboost_reg_pred_res = sk_gradboost_reg.predict(X2_test_scaled)
sk_gradboost_reg_r2 = r2_score(y2_test, sk_gradboost_reg_pred_res)
sk_gradboost_reg_mae = mean_absolute_error(y2_test, sk_gradboost_reg_pred_res)
sk_gradboost_reg_mse = mean_squared_error(y2_test, sk_gradboost_reg_pred_res)

print(f'sk gradboost regressor r2: {sk_gradboost_reg_r2}')
print(f'sk gradboost regressor mae: {sk_gradboost_reg_mae}')
print(f'sk gradboost regressor mse: {sk_gradboost_reg_mse}')

sk gradboost regressor r2: 0.8747621800409676
sk gradboost regressor mae: 4131.700034443791
sk gradboost regressor mse: 35294044.26691111


3. Подбор гиперпараметров

In [69]:
param_grid_reg = {
    'n_estimators': [10, 50, 100, 150],
    'max_depth': [None, 4, 10, 20, 30],
    'learning_rate': [0.1, 0.25, 0.5, 0.75]
}

gradboost_reg = GradientBoostingRegressor()
grid_search_reg = GridSearchCV(gradboost_reg, param_grid_reg, cv=5, scoring='r2')
grid_search_reg.fit(X2_train, y2_train)
best_gradboost_reg = grid_search_reg.best_estimator_

print("Лучшие параметры для регрессии:", grid_search_reg.best_params_)

y_pred_reg = best_gradboost_reg.predict(X2_test)
r2 = r2_score(y2_test, y_pred_reg)
mae = mean_absolute_error(y2_test, y_pred_reg)
mse = mean_squared_error(y2_test, y_pred_reg)

print(f"r2: {r2}")
print("mae:", mae)
print("mse:", mse)

Лучшие параметры для регрессии (CO2): {'learning_rate': 0.25, 'max_depth': 4, 'n_estimators': 100}
r2: 0.9014751764087997
mae: 3895.6865540068197
mse: 27765889.61988424


### Выводы

Введенные улучшения - формирование новых признаков и подбор параметров - позволили несколько улучшить результат. От масштабирования ощутимого улучшения не произошло.

## 4. Имплементация алгоритма машинного обучения 

### Самостоятельная имплементация алгоритмов машинного обучения для классификации и регрессии

Реализация для классификации

In [65]:
from sklearn.tree import DecisionTreeRegressor

class GradientBoostingClassifierCustom:
    def __init__(self, n_estimators=100, max_depth=4, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.models = []
        self.initial_prediction = None
        self.classes_ = None

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.classes_ = np.unique(y)
        self.models = {cls: [] for cls in self.classes_}
        self.initial_prediction = {cls: np.log(np.sum(y == cls) / len(y)) for cls in self.classes_}
        y_one_hot = np.array([[1 if yi == cls else 0 for cls in self.classes_] for yi in y])

        for cls_idx, cls in enumerate(self.classes_):
            residuals = y_one_hot[:, cls_idx] - self._sigmoid(self.initial_prediction[cls])
            for _ in range(self.n_estimators):
                tree = DecisionTreeRegressor(max_depth=self.max_depth)
                tree.fit(X, residuals)
                predictions = tree.predict(X)
                residuals -= self.learning_rate * predictions
                self.models[cls].append(tree)

    def predict(self, X):
        X = np.array(X)
        logits = {cls: np.full(X.shape[0], self.initial_prediction[cls]) for cls in self.classes_}
        for cls in self.classes_:
            for tree in self.models[cls]:
                logits[cls] += self.learning_rate * tree.predict(X)

        exp_logits = np.exp(np.array(list(logits.values())).T)
        probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

        return np.array(self.classes_)[np.argmax(probabilities, axis=1)]

    def _sigmoid(self, logits):
        return 1 / (1 + np.exp(-logits))

Реализация для регресии

In [66]:
class GradientBoostingRegressorCustom:
    def __init__(self, n_estimators=100, max_depth=4, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.models = []
        self.initial_prediction = None

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.models = []
        self.initial_prediction = np.mean(y)
        residuals = y - self.initial_prediction

        for _ in range(self.n_estimators):
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            predictions = tree.predict(X)
            residuals -= self.learning_rate * predictions
            self.models.append(tree)

    def predict(self, X):
        X = np.array(X)
        predictions = np.full(X.shape[0], self.initial_prediction)
        for tree in self.models:
            predictions += self.learning_rate * tree.predict(X)
        return predictions

### Обучение имплементированных моделей

In [70]:
custom_gradboost_clf = GradientBoostingClassifierCustom(n_estimators=10, max_depth=4, learning_rate=0.1)
custom_gradboost_clf.fit(X1_train, y1_train)
custom_gradboost_clf_pred_res = custom_gradboost_clf.predict(X1_test)
custom_gradboost_clf_accuracy = accuracy_score(y1_test, custom_gradboost_clf_pred_res)
custom_gradboost_clf_precision = precision_score(y1_test, custom_gradboost_clf_pred_res)
custom_gradboost_clf_recall = recall_score(y1_test, custom_gradboost_clf_pred_res)
custom_gradboost_clf_f1 = f1_score(y1_test, custom_gradboost_clf_pred_res)

print(f'custom gradboost classifier accuracy: {custom_gradboost_clf_accuracy:}')
print(f'custom gradboost classifier precision: {custom_gradboost_clf_precision:}')
print(f'custom gradboost classifier recall: {custom_gradboost_clf_recall:}')
print(f'custom gradboost classifier f1: {custom_gradboost_clf_f1:}')

custom gradboost classifier accuracy: 0.6141732283464567
custom gradboost classifier precision: 0.7014925373134329
custom gradboost classifier recall: 0.618421052631579
custom gradboost classifier f1: 0.6573426573426574


In [73]:
custom_gradboost_reg = GradientBoostingRegressorCustom(n_estimators=10, max_depth=4, learning_rate=0.1)
custom_gradboost_reg.fit(X2_train, y2_train)
custom_gradboost_reg_pred_res = custom_gradboost_reg.predict(X2_test)
custom_gradboost_reg_r2 = r2_score(y2_test, custom_gradboost_reg_pred_res)
custom_gradboost_reg_mae = mean_absolute_error(y2_test, custom_gradboost_reg_pred_res)
custom_gradboost_reg_mse = mean_squared_error(y2_test, custom_gradboost_reg_pred_res)

print(f'custom gradboost regressor r2: {custom_gradboost_reg_r2}')
print(f'custom gradboost regressor mae: {custom_gradboost_reg_mae}')
print(f'custom gradboost regressor mse: {custom_gradboost_reg_mse}')

custom gradboost regressor r2: 0.6980158026327707
custom gradboost regressor mae: 6964.097443083268
custom gradboost regressor mse: 85104033.53614041


### Выводы

Полученные результаты даже превосходят результаты встроенных моделей и близки к результатам при использовании оптимизаций, что говорит об отсутствии проблем с реализацией и использованием датасета с этими алгоритмами, и, по-видимому, довольно удачном выборе параметров.

### Обучение имплементированных моделей в улучшенном бейзлайне

#### Задача классификации

Проверим, как повлияет введение новых признаков

In [74]:
custom_gradboost_clf = GradientBoostingClassifierCustom(n_estimators=10, max_depth=4, learning_rate=0.1)
custom_gradboost_clf.fit(X1_train_new, y1_train_new)
custom_gradboost_clf_pred_res = custom_gradboost_clf.predict(X1_test_new)
custom_gradboost_clf_accuracy = accuracy_score(y1_test_new, custom_gradboost_clf_pred_res)
custom_gradboost_clf_precision = precision_score(y1_test_new, custom_gradboost_clf_pred_res)
custom_gradboost_clf_recall = recall_score(y1_test_new, custom_gradboost_clf_pred_res)
custom_gradboost_clf_f1 = f1_score(y1_test_new, custom_gradboost_clf_pred_res)

print(f'custom gradboost classifier accuracy: {custom_gradboost_clf_accuracy:}')
print(f'custom gradboost classifier precision: {custom_gradboost_clf_precision:}')
print(f'custom gradboost classifier recall: {custom_gradboost_clf_recall:}')
print(f'custom gradboost classifier f1: {custom_gradboost_clf_f1:}')

custom gradboost classifier accuracy: 0.6614173228346457
custom gradboost classifier precision: 0.6896551724137931
custom gradboost classifier recall: 0.7894736842105263
custom gradboost classifier f1: 0.736196319018405


Попробуем добавить к формированию новых признаков масштабирование

In [75]:
custom_knn_clf = GradientBoostingClassifierCustom(n_estimators=10, max_depth=4, learning_rate=0.1)
custom_knn_clf.fit(X1_train_new_scaled, y1_train_new)
custom_knn_clf_pred_res = custom_knn_clf.predict(X1_test_new_scaled)
custom_knn_clf_accuracy = accuracy_score(y1_test_new, custom_knn_clf_pred_res)
custom_gradboost_clf_precision = precision_score(y1_test_new, custom_gradboost_clf_pred_res)
custom_gradboost_clf_recall = recall_score(y1_test_new, custom_gradboost_clf_pred_res)
custom_gradboost_clf_f1 = f1_score(y1_test_new, custom_gradboost_clf_pred_res)

print(f'custom gradboost classifier accuracy: {custom_gradboost_clf_accuracy:}')
print(f'custom gradboost classifier precision: {custom_gradboost_clf_precision:}')
print(f'custom gradboost classifier recall: {custom_gradboost_clf_recall:}')
print(f'custom gradboost classifier f1: {custom_gradboost_clf_f1:}')

custom gradboost classifier accuracy: 0.6614173228346457
custom gradboost classifier precision: 0.6896551724137931
custom gradboost classifier recall: 0.7894736842105263
custom gradboost classifier f1: 0.736196319018405


Попробуем добавить к формированию новых признаков подбор параметров

In [76]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"accuracy": 0}

n_estimators_variants = [10, 50, 100, 150]
max_depth_variants = [None, 4, 10, 20, 30]
learning_rate_variants = [0.1, 0.25, 0.5, 0.75]

for n_estimators, max_depth, learning_rate in product(n_estimators_variants, max_depth_variants, learning_rate_variants):
    custom_tree_clf = GradientBoostingClassifierCustom(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate)
    custom_tree_clf.fit(X1_train_new, y1_train_new)
    pred_res = custom_tree_clf.predict(X1_test_new)

    accuracy = accuracy_score(y1_test_new, pred_res)
    precision = precision_score(y1_test_new, pred_res)
    recall = recall_score(y1_test_new, pred_res)
    f1 = f1_score(y1_test_new, pred_res)

    if accuracy > best_metrics_classification["accuracy"]:
        best_params_classification = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate
        }
        best_metrics_classification = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }
        
print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

Лучшие параметры для классификации: {'n_estimators': 10, 'max_depth': 10, 'learning_rate': 0.25}
Метрики для классификации: {'accuracy': 0.6929133858267716, 'precision': 0.7228915662650602, 'recall': 0.7894736842105263, 'f1': 0.7547169811320756}


#### Задача регрессии

Проверим, как повлияет масштабирование

In [77]:
custom_gradboost_reg = GradientBoostingRegressorCustom(n_estimators=10, max_depth=4, learning_rate=0.1)
custom_gradboost_reg.fit(X2_train_scaled, y2_train)
custom_gradboost_reg_pred_res = custom_gradboost_reg.predict(X2_test_scaled)
custom_gradboost_reg_r2 = r2_score(y2_test, custom_gradboost_reg_pred_res)
custom_gradboost_reg_mae = mean_absolute_error(y2_test, custom_gradboost_reg_pred_res)
custom_gradboost_reg_mse = mean_squared_error(y2_test, custom_gradboost_reg_pred_res)

print(f'custom gradboost regressor r2: {custom_gradboost_reg_r2}')
print(f'custom gradboost regressor mae: {custom_gradboost_reg_mae}')
print(f'custom gradboost regressor mse: {custom_gradboost_reg_mse}')

custom gradboost regressor r2: 0.6872581800723508
custom gradboost regressor mae: 7020.928378477048
custom gradboost regressor mse: 88135705.65386319


Проверим, как повлияет подбор параметров

In [79]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"r2": 0}

n_estimators_variants = [10, 50, 100, 150]
max_depth_variants = [None, 4, 10, 20, 30]
learning_rate_variants = [0.1, 0.25, 0.5, 0.75]

for n_estimators, max_depth, learning_rate in product(n_estimators_variants, max_depth_variants, learning_rate_variants):
    custom_tree_clf = GradientBoostingRegressorCustom(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate)
    custom_tree_clf.fit(X2_train, y2_train)
    pred_res = custom_tree_clf.predict(X2_test)

    r2 = r2_score(y2_test, pred_res)
    mae = mean_absolute_error(y2_test, pred_res)
    mse = mean_squared_error(y2_test, pred_res)

    if r2 > best_metrics_classification["r2"]:
        best_params_classification = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate
        }
        best_metrics_classification = {
            "r2": r2,
            "mae": mae,
            "mse": mse
        }
        
print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

Лучшие параметры для классификации: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.25}
Метрики для классификации: {'r2': 0.899999974470221, 'mae': 3984.65861382469, 'mse': 28181625.397941273}


### Выводы

Как и для встроенных моделей, удалось улучшить результаты работы моделей.
В целом, для регресии результаты моделей в улучшенных бейзлайнах схожи, для классификации метрики у собственной модели даже лучше.